# 2. Task 1: Text preprocessing & vectorization

We build 4 vectorizations of our trail descriptions:
1. TF-IDF (classical)
2. FastText averaged embeddings (embedding-based)
3. LDA topic vectors (topic modeling)
4. BERT sentence embeddings (contextual)

In [1]:
# pip install spacy gensim sentence-transformers scikit-learn
# python -m spacy download en_core_web_sm

import pickle
from pathlib import Path
import pandas as pd
import numpy as np

# Connect to google drive
from google.colab import drive
colab = True

# 2. Set the directory path first
if colab:
    drive.mount('/content/drive')
    DIRECTORY = "/content/drive/MyDrive/YEAR 4 - SPRING/ML Applications/ml-applications-project-repo" # Cambia al tuyo
else:
    DIRECTORY = ".."

df = pd.read_pickle(DIRECTORY + "/data/trails.pkl")
print(f"{len(df)} trails loaded")

# we'll work with the description as the main text document
texts_raw = df["description"].tolist()
ids = df["tour_id"].tolist()

Mounted at /content/drive
5642 trails loaded


In [21]:
# Run this if the files already exist
TOKENIZED_CACHE = Path(DIRECTORY + "/data/tokenized.pkl")
TEXTS_CLEAN_CACHE = Path(DIRECTORY + "/data/texts_clean.pkl")

if TOKENIZED_CACHE.exists() and TEXTS_CLEAN_CACHE.exists():
    with open(TOKENIZED_CACHE, "rb") as f:
        tokenized = pickle.load(f)
    with open(TEXTS_CLEAN_CACHE, "rb") as f:
        texts_clean = pickle.load(f)
    print(f"Loaded preprocessed texts from cache ({len(tokenized)} docs)")
else:
    print("Preprocessing texts...")
    tokenized = [preprocess(t) for t in texts_raw]
    texts_clean = [" ".join(tokens) for tokens in tokenized]

Loaded preprocessed texts from cache (5642 docs)


## Step 1 — Text preprocessing pipeline

Using SpaCy: tokenize, lowercase, remove stopwords and punctuation, lemmatize.

In [2]:
import spacy

# load English model: python -m spacy download en_core_web_sm
nlp = spacy.load("en_core_web_sm", disable=["parser", "ner"])

def preprocess(text: str) -> list[str]:
    """Return list of clean lemmas from a text"""
    doc = nlp(text.lower())
    tokens = [
        token.lemma_
        for token in doc
        if not token.is_stop       # stopwords
        and not token.is_punct     # punctuation
        and not token.is_space     # whitespace
        and len(token.lemma_) > 2  # skip short tokens
    ]
    return tokens

# process all descriptions
print("Preprocessing texts...")
tokenized = [preprocess(t) for t in texts_raw]

# quick sanity check
print("Example:", tokenized[0][:10])

Preprocessing texts...
Example: ['moderate', '20.1', 'mile', 'mountain', 'biking', 'route', 'near', 'cabezón', 'sal', 'offer']


In [20]:
texts_clean = [" ".join(tokens) for tokens in tokenized]

# save both tokenized (for LDA/FastText) and clean strings (for TF-IDF/BERT)
with open(DIRECTORY + "/data/tokenized.pkl", "wb") as f:
    pickle.dump(tokenized, f)

with open(DIRECTORY + "/data/texts_clean.pkl", "wb") as f:
    pickle.dump(texts_clean, f)

print(f"Saved {len(tokenized)} tokenized docs and {len(texts_clean)} clean strings")

Saved 5642 tokenized docs and 5642 clean strings


## Step 2a — TF-IDF (classical approach)

In [4]:
from sklearn.feature_extraction.text import TfidfVectorizer

# max_features limits vocabulary size
tfidf = TfidfVectorizer(max_features=5000)
tfidf_matrix = tfidf.fit_transform(texts_clean)

print("TF-IDF shape:", tfidf_matrix.shape)

# save both the matrix and the fitted vectorizer
import scipy.sparse as sp
sp.save_npz(DIRECTORY + "/data/tfidf_matrix.npz", tfidf_matrix)
with open(DIRECTORY + "/data/tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

print("Saved TF-IDF")

TF-IDF shape: (5642, 5000)
Saved TF-IDF


In [5]:
# top terms per sport
feature_names = tfidf.get_feature_names_out()
tfidf_dense = tfidf_matrix.toarray()

for sport in df["sport"].unique():
    mask = df["sport"].values == sport
    sport_mean = tfidf_dense[mask].mean(axis=0)
    top_idx = sport_mean.argsort()[-10:][::-1]
    top_terms = [feature_names[i] for i in top_idx]
    print(f"{sport}: {top_terms}")

mtb: ['mountain', 'biking', 'bike', 'offer', 'mile', 'route', 'moderate', 'view', 'terrain', 'loop']
hike: ['hike', 'trail', 'circular', 'moderate', 'mile', 'view', 'panoramic', 'explore', 'easy', 'del']
jogging: ['jog', 'jogging', 'mile', 'view', 'trail', 'difficult', 'route', 'offer', 'foot', 'moderate']
racebike: ['road', 'cycling', 'route', 'mile', 'difficult', 'offer', 'climb', 'moderate', 'view', 'cycle']
touringbicycle: ['touring', 'cycle', 'cycling', 'route', 'mile', 'difficult', 'loop', 'offer', 'explore', 'moderate']
mtb_easy: ['gravel', 'route', 'town', 'track', 'kilometer', 'day', 'cross', 'bike', 'take', 'reach']


## Step 2b — FastText embeddings (embedding-based approach)

We train FastText on our corpus and average word vectors to get one vector per trail.

In [18]:
from gensim.models import FastText

# train FastText on our tokenized descriptions
# ignore rare words
print("Training FastText...")
ft_model = FastText(sentences=tokenized, vector_size=100, window=5, min_count=2, epochs=10, seed=42, workers=1)
print("Done")

# save the model
ft_model.save(DIRECTORY + "/data/fasttext_model")

Training FastText...
Done


In [12]:
def doc_vector_fasttext(tokens: list[str], model, doc_id=None) -> np.ndarray:
    """Average the word vectors of a document. Warns if no tokens are found in vocabulary."""
    vecs = [model.wv[w] for w in tokens if w in model.wv]
    if not vecs:
        label = f"doc_id={doc_id}" if doc_id is not None else "unknown"
        print(f"  WARNING: no vocabulary match for {label}, returning zero vector")
        return np.zeros(model.vector_size)
    return np.mean(vecs, axis=0)

fasttext_matrix = np.array([doc_vector_fasttext(t, ft_model, doc_id=ids[i])
                             for i, t in enumerate(tokenized)])

zero_mask = ~fasttext_matrix.any(axis=1)
if zero_mask.sum() > 0:
    print(f"\n{zero_mask.sum()} documents returned zero vectors — check preprocessing for these trails.")
else:
    print("All documents have non-zero FastText vectors.")

fasttext_matrix = np.array([doc_vector_fasttext(t, ft_model) for t in tokenized])
print("FastText matrix shape:", fasttext_matrix.shape)

np.save(DIRECTORY + "/data/fasttext_matrix.npy", fasttext_matrix)
print("Saved FastText vectors")

All documents have non-zero FastText vectors.
FastText matrix shape: (5642, 100)
Saved FastText vectors


## Step 2c — LDA topic model (topic-modeling approach)

We need to choose the number of topics. We try several values and pick the one
with the best coherence score.

In [13]:
from gensim import corpora
from gensim.models import LdaModel, CoherenceModel

# build dictionary and BoW corpus
dictionary = corpora.Dictionary(tokenized)
dictionary.filter_extremes(no_below=5, no_above=0.5)  # remove too rare and too common
corpus_bow = [dictionary.doc2bow(tokens) for tokens in tokenized]

print(f"Vocabulary size after filtering: {len(dictionary)}")

Vocabulary size after filtering: 2281


In [17]:
# try different number of topics and compute coherence
topic_range = [5, 10, 15, 20, 25, 30]
coherence_scores = []

import time

COHERENCE_CACHE = DIRECTORY / "data" / "coherence_scores.pkl"

if COHERENCE_CACHE.exists():
    with open(COHERENCE_CACHE, "rb") as f:
        coherence_scores = pickle.load(f)
    print(f"Loaded coherence scores from cache: {list(zip(topic_range, coherence_scores))}")
else:
    for i, n_topics in enumerate(topic_range, 1):
        t0 = time.time()
        print(f"[{i}/{len(topic_range)}] Training LDA with {n_topics} topics...", end=" ", flush=True)
        lda = LdaModel(corpus=corpus_bow, id2word=dictionary,
                       num_topics=n_topics, random_state=42,
                       passes=10, alpha="auto")
        cm = CoherenceModel(model=lda, texts=tokenized,
                            dictionary=dictionary, coherence="c_v")
        score = cm.get_coherence()
        coherence_scores.append(score)
        print(f"coherence={score:.4f} ({time.time()-t0:.1f}s)")

    with open(COHERENCE_CACHE, "wb") as f:
        pickle.dump(coherence_scores, f)
    print("Coherence scores cached.")

TypeError: unsupported operand type(s) for /: 'str' and 'str'

In [ ]:
import matplotlib.pyplot as plt

plt.plot(topic_range, coherence_scores, marker="o")
plt.xlabel("Number of topics")
plt.ylabel("Coherence (c_v)")
plt.title("LDA coherence vs number of topics")
plt.tight_layout()
plt.show()

best_n = topic_range[np.argmax(coherence_scores)]
print(f"Best number of topics: {best_n}")

In [ ]:
# train final LDA with the best number of topics
lda_model = LdaModel(corpus=corpus_bow, id2word=dictionary,
                     num_topics=best_n, random_state=42,
                     passes=15, alpha="auto")

# show top words per topic
for i, topic in lda_model.show_topics(num_topics=best_n, num_words=8, formatted=False):
    words = [w for w, _ in topic]
    print(f"Topic {i}: {words}")

In [ ]:
# convert each document to a topic distribution vector
def doc_to_topic_vector(bow, model, n_topics):
    """Returns a dense vector of topic probabilities for a document."""
    topic_dist = dict(model.get_document_topics(bow, minimum_probability=0.0))
    return np.array([topic_dist.get(i, 0.0) for i in range(n_topics)])

lda_matrix = np.array([doc_to_topic_vector(bow, lda_model, best_n)
                        for bow in corpus_bow])
print("LDA matrix shape:", lda_matrix.shape)

np.save(DIRECTORY + "/data/lda_matrix.npy", lda_matrix)
lda_model.save(DIRECTORY + "/data/lda_model")
dictionary.save(DIRECTORY + "/data/lda_dictionary")
print("Saved LDA vectors")

## Step 2d — BERT sentence embeddings (contextual approach)

We use `sentence-transformers` with a lightweight multilingual model.
This model handles English descriptions and would also work with Spanish ones.

In [ ]:
from sentence_transformers import SentenceTransformer

# paraphrase-MiniLM
bert_model = SentenceTransformer("paraphrase-MiniLM-L6-v2")

# encode raw descriptions
bert_matrix = bert_model.encode(texts_raw, batch_size=32, show_progress_bar=True)

print("BERT matrix shape:", bert_matrix.shape)
np.save(DIRECTORY + "/data/bert_matrix.npy", bert_matrix)
print("Saved BERT vectors")

## PCA visualization of all 4 vectorizations

Trails cluster by sport?

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt

matrices = {
    "TF-IDF": tfidf_matrix.toarray(),
    "FastText": fasttext_matrix,
    "LDA": lda_matrix,
    "BERT": bert_matrix,
}

sports = df["sport"].values
sport_labels = {s: i for i, s in enumerate(df["sport"].unique())}
colors = [sport_labels[s] for s in sports]

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

for ax, (name, matrix) in zip(axes, matrices.items()):
    pca = PCA(n_components=2, random_state=42)
    coords = pca.fit_transform(matrix)
    ax.scatter(coords[:, 0], coords[:, 1], c=colors, alpha=0.4, s=5, cmap="tab10")
    ax.set_title(name)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")

plt.suptitle("PCA of vectorizations (colour = sport)")
plt.tight_layout()
plt.savefig(DIRECTORY + "/data/pca_comparison.png", dpi=150)
plt.show()